# 02 - EDX-Auswertung (Nanopore)

Energiedispersive Roentgenspektroskopie am selben Probenbereich wie Notebook 01.
Wir bestimmen Elementlinien, fitten sie und vergleichen zwei Wege zur
Linienintensitaet: **Modellfit** und **Untergrundfenster**.

**Voraussetzung:** `00_setup_check.ipynb` lief fehlerfrei durch.

Dokumentation: <https://hyperspy.org/exspy/user_guide/eds.html>

In [ ]:
# Interaktive Plots (zoomen, Spektrum je Bildpunkt anklicken).
# Falls die Plots weiss bleiben oder gar nichts erscheint:
# diese Zeile durch  %matplotlib inline  ersetzen und den Kernel neu starten.
%matplotlib widget

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d

import hyperspy.api as hs
import exspy  # muss importiert sein, sonst kennt HyperSpy die EELS-/EDX-Signaltypen nicht

# Findet die Messdaten unabhaengig vom Betriebssystem (siehe workshop_data.py)
from workshop_data import load, load_standards

print("HyperSpy", hs.__version__, "| exspy", exspy.__version__)

## 1. Daten laden

In [ ]:
edx = load("edx_si", signal_type="EDS_TEM")
edx

## 2. ADF-Bilder

`ADF Image (SI Survey)` ist die Uebersichtsaufnahme vor der Messung,
`ADF Image` das gleichzeitig zum Spektrenbild aufgenommene Signal.

In [ ]:
load("adf_survey").plot()

In [ ]:
load("adf").plot()

## 3. Elemente festlegen

`set_elements` sagt, welche Elemente vorkommen, `add_lines` waehlt dazu
automatisch die passenden Roentgenlinien (K-alpha usw.) fuer die
Beschleunigungsspannung dieser Messung aus.

In [ ]:
edx_binned = edx.rebin(scale=[2, 2, 1])

edx_binned.set_elements(["Si", "N", "O"])
edx_binned.add_lines()

print(edx_binned.metadata.Sample.xray_lines)
edx_binned.plot(True)  # True = Linienmarkierungen einzeichnen

## 4. Modellfit

Zuerst am **Summenspektrum** (alle Bildpunkte aufaddiert): rauscharm, schnell,
gut zum Pruefen, ob die Linienauswahl stimmt.

In [ ]:
edx_sum = edx_binned.sum()

m_summed = edx_sum.create_model()
m_summed.fit_background()
m_summed.fit()
m_summed.plot()

Und jetzt jeder Bildpunkt einzeln.

**Hinweis:** Im urspruenglichen Notebook stand hier `m.fit()`, das nur den
*aktuell ausgewaehlten* Bildpunkt fittet - `plot_results()` haette danach fast
ueberall Startwerte statt Fitergebnisse gezeigt. `multifit()` geht ueber alle
Bildpunkte. Das dauert entsprechend laenger.

In [ ]:
m = edx_binned.create_model()
m.fit_background()
m.multifit()

In [ ]:
m.plot()

In [ ]:
m.plot_results()

## 5. Der zweite Weg: Untergrundfenster

Statt zu modellieren kann man die Linienintensitaet auch direkt integrieren und
den Untergrund aus je einem Fenster links und rechts der Linie abziehen.
Schneller und anschaulicher, aber empfindlich gegen ueberlappende Linien.

`estimate_background_windows` schlaegt die Fenster automatisch vor.

In [ ]:
bw = edx_sum.estimate_background_windows(line_width=[5.0, 2.0])

print("Je Zeile eine Linie: [links_von, links_bis, rechts_von, rechts_bis] in keV")
print(bw)

edx_sum.plot(background_windows=bw)

In [ ]:
intensitaeten = edx_binned.get_lines_intensity(background_windows=bw, plot_result=True)

## Aufgabe: Fenster von Hand setzen

Die automatische Schaetzung ist nur ein Vorschlag. Schau dir den Plot oben genau an -
liegt ein Fenster auf einer benachbarten Linie oder auf einer Flanke, wird der
Untergrund falsch geschaetzt und die Intensitaet stimmt nicht.

Unten stehen die geschaetzten Werte als Startpunkt. Aendere die Zahlen, fuehre die
Zelle aus, schau dir den Plot an - und vergleiche danach die Intensitaetskarten
mit denen von oben.

Die Reihenfolge der Zeilen entspricht `edx_binned.metadata.Sample.xray_lines`.

In [ ]:
print("Reihenfolge der Zeilen:", edx_binned.metadata.Sample.xray_lines)

bw_manuell = bw.copy()

# Hier von Hand anpassen, z.B.:
# bw_manuell[0] = [1.55, 1.63, 1.83, 1.91]   # [links_von, links_bis, rechts_von, rechts_bis]

edx_sum.plot(background_windows=bw_manuell)

In [ ]:
intensitaeten_manuell = edx_binned.get_lines_intensity(
    background_windows=bw_manuell, plot_result=True
)

## Aufgaben

1. Vergleiche fuer eine Linie die Intensitaet aus dem Modellfit (Abschnitt 4)
   mit der aus den Untergrundfenstern (Abschnitt 5). Woher kommt der Unterschied?
2. Aendere `line_width=[5.0, 2.0]` auf `[10.0, 5.0]`. Was passiert mit den
   Fenstern und warum ist das bei dicht beieinanderliegenden Linien gefaehrlich?
3. Die Probe ist eine Si-N-Membran. Passt das Verhaeltnis der gemessenen
   Intensitaeten zu dem, was du erwartest? Was fehlt noch zu einer echten
   Quantifizierung?